# Live trading Turtle Mean reversion 30 Min

Este cuaderno implementa la versión en vivo realizada en backtest en el cuaderno en Jupyter. Guarda el Journal de la jornada con el estado inicial, el monitoreo de MAE/R para cada posición y los cierres junto con el PnL. A diferencia del backtesting este cuaderno resume la ejecución a través de los módulos dirigidos por eventos:
* streaming.py : Conecta el Websocket de precios en vivo de IBKR para monitorear precios y data de usuario.
* event/event.py : Diccionario de eventos que incluye eventos de mercado (nuevos ticks) señales, calces , cierres y eventos de portafolio
* portfolio: implementa la lógica de portafolio , administración de posiciones , sizing , upsizing , downsizing, riesgo a nivel portafolio (si  multiactivos) estado de la cuenta (en local) , ordenes enviadas sin ejecución , niveles de calce etc. Es el módulo más complejo.
* strategy : Guarda la lógica de las estrategias y emite eventos señal al adminostrador de portafolio
* execution : Lanza ordenes a IBKR, Order Management System, reconcilia ordenes y posiciones , administra stops y condicionales. Administra la ejecución y mensajes de IBKR
* trading : Conecta todos los módulos y administra la cola de eventos.

Esta construcción originalmente se implementó para forex y acciones y se ha acondicionado particularmente para la prueba técnica propuesta por Sun Valley.  

# Imports y conexiones con IBKR


In [ ]:
# pip install ib_async pandas numpy
import json, logging, os, queue, sys, threading, time
from datetime import datetime, timezone
from decimal import Decimal
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from ib_async import IB, util

# ── Los módulos del motor. Toda la lógica vive aquí, no en el notebook ──────
from qsCrypto.ibkr import config
from qsCrypto.data.streaming import IBKRPriceHandler
from qsCrypto.strategy.strategy import MeanReversionFadeStrategy
from qsCrypto.portfolio.portfolio import IBKRFuturesPortfolio
from qsCrypto.execution.execution import create_execution_handler
from qsCrypto.trading.trading import trade

JOURNAL_DIR = ROOT / "journals"; JOURNAL_DIR.mkdir(exist_ok=True)
STATE_DIR = ROOT / "output"; STATE_DIR.mkdir(exist_ok=True)

# ── Logging del motor a ARCHIVO, no a print() ───────────────────────────────
# Todo qsCrypto/* (streaming, portfolio, trading, execution) usa
# `logging.getLogger(__name__)` — nunca `print`. Sin un handler configurado
# esos logs (incluidos los CRITICAL de "sin stop detectado" o "feed
# obsoleto") se pierden en silencio: `trading.py` solo llama a
# `logging.config.fileConfig` dentro de su bloque `if __name__ == "__main__"`,
# que jamás se ejecuta al importar el módulo desde un notebook. Un
# `StreamHandler` a la salida del notebook no sirve tampoco: `monitor_loop()`
# hace `clear_output(wait=True)` cada 30s y se lo llevaría por delante. El
# archivo es lo único que sobrevive a eso — ábrelo con
# `tail -f output/live_engine.log` en una terminal aparte mientras corre.
LOG_FILE = STATE_DIR / "live_engine.log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s: %(message)s",
    handlers=[logging.FileHandler(LOG_FILE)],
    force=True,   # pisa cualquier configuración que ib_async/Jupyter deje puesta
)
print(f"  Log del motor: {LOG_FILE} (tail -f {LOG_FILE} en una terminal)")

# Para el cuaderno
def _in_jupyter():
    try:
        return get_ipython().__class__.__name__ == "ZMQInteractiveShell"
    except NameError:
        return False

if _in_jupyter():
    util.startLoop()          # sin esto connect() muere: el kernel ya tiene bucle

ib = IB()
# RequestTimeout viene a 0 y ib_async hace `if timeout:` — CERO SIGNIFICA SIN
# LÍMITE. Una petición sin respuesta colgaría el kernel para siempre y dejaría
# la posición sin vigilar.
ib.RequestTimeout = 20

IB_ERRORS = [] # Manejo de errores devueltos por la API
def _on_ib_error(reqId, code_, msg, contract):
    if code_ not in (2104, 2106, 2158, 2119):
        IB_ERRORS.append((datetime.now(timezone.utc).isoformat(), reqId, code_, msg))
        print(f"  [TWS {code_}] {msg}")

ib.connect(config.IB_HOST, config.IB_PORT, clientId=config.IB_CLIENT_ID,
           timeout=15, readonly=False)
ib.errorEvent += _on_ib_error

ACCOUNT = config.IB_ACCOUNT or ib.managedAccounts()[0]
ACCOUNTS = {a: ("master" if a.upper().startswith("DF") else "trading")
            for a in ib.managedAccounts()}
summary = {v.tag: v.value for v in ib.accountSummary(ACCOUNT)
           if v.tag in ("NetLiquidation", "AvailableFunds", "BuyingPower")}
EQUITY = Decimal(summary.get("NetLiquidation", "100000"))

print(f"  Conectado a {config.IB_HOST}:{config.IB_PORT} (clientId={config.IB_CLIENT_ID})")
print(f"  Cuenta operada : {ACCOUNT}")
print(f"  Cuentas        : {ACCOUNTS}")
print(f"  Equity         : {EQUITY:,.2f}")


  Log del motor: /Users/diegoochoa/Projects/Intraday_MR_BTC/output/live_engine.log (tail -f /Users/diegoochoa/Projects/Intraday_MR_BTC/output/live_engine.log en una terminal)
  Conectado a 127.0.0.1:7496 (clientId=17)
  Cuenta operada : DU496961
  Cuentas        : {'DF496956': 'master', 'DU496957': 'trading', 'DU496958': 'trading', 'DU496959': 'trading', 'DU496960': 'trading', 'DU496961': 'trading'}
  Equity         : 353,172.89


  [TWS 162] Mensaje de error de los servicios de datos de mercado hist\u00f3ricos:API historical data query cancelled: 363
  [TWS 162] Mensaje de error de los servicios de datos de mercado hist\u00f3ricos:API historical data query cancelled: 366


## 1. Qué dijo el backtest, antes de arrancar nada

Esta implementación reproduce la estrategia de `MeanReversion_IBKR_CME_Bitcoin.ipynb`. Ese
notebook se dictamina a sí mismo **8 de 12 criterios** y concluye literalmente *"No desplegar
capital con esta evidencia"*. Los cuatro criterios que falla:

| Criterio | Exigido | Medido |
|---|---|---|
| Muestra | ≥ 2 años | **1,34 años** |
| Folds OOS | ≥ 8 | **6** (el t-test no tiene potencia) |
| Apuestas independientes | ≥ 8 | **n_eff = 1,00** |
| Deflated Sharpe | > 0,95 | **0,374** |

Sobre el DSR el backtest es explícito: el Sharpe seleccionado **no supera lo que cabría esperar
del mejor de esa rejilla sin ningún edge**. Con `n_eff = 1` un drawdown del 50% no es cola, es el
escenario central.

**Esto es un ejercicio técnico sobre cuenta paper.** El objetivo es que la estrategia corra por
los módulos dirigidos por eventos y que el MAE/MFE sea medible en vivo, no ganar dinero.

### El cierre temprano por MAE no puede dispararse

El backtest calibra **MAE\* = 1,55 N**, y opera un stop de **1 N**. Como `R = stop_n · N`, el
umbral de MAE cae **fuera** del stop: el stop del bróker llega siempre antes. El MAE se mide, se
registra y se muestra en el monitor, pero con `STOP_N = 1.0` **no cierra nada**. Subir `STOP_N`
haría que quepa, a cambio de abandonar el único punto de la curva de sensibilidad con Sharpe alto
(1,86 a 1N frente a 0,31 a 3N). El parámetro queda expuesto abajo.

In [ ]:
# ─────────── PARÁMETROS, DECLARADOS ANTES DE ABRIR NADA ───────────
SYMBOLS       = ["MBT"]      # Micro Bitcoin CME, 0.1 BTC/contrato
EXCHANGE      = "CME"
CURRENCY      = "USD"
BAR_SIZE      = "30 mins"    # la granularidad del backtest
USE_RTH       = True
WHAT_TO_SHOW  = "TRADES"

# Señal (celda 4 y 24 del backtest)
ENTRY_BARS    = 20           # canal Donchian de entrada
ATR_PERIOD    = 20           # N = EMA-20 del True Range
SIGNAL_MODE   = "fade"       # se opera CONTRA la ruptura
REQUIRE_CONF  = False         # la barra t+1 debe cerrar dentro del canal
CONFIRM_MODE  = "inside"
ALLOW_SHORT   = True

# Riesgo y barreras
RISK_PER_UNIT = 0.01         # 1% del equity por unidad
STOP_N        = 1.0          # R = 1N. Ver §1: con esto el cierre por MAE no cabe.
ADD_N         = 0.5          # escalera de adds cada ½N
MAX_UNITS     = 4
MAX_HOLD_BARS = 10           # barrera vertical: 10 barras de 30 min = 5 h
TP_MODE       = "mid_channel"  # objetivo en el centro del canal, congelado en t
MAX_GROSS_LEV = 20.0
MAX_DIR_UNITS = 12
MAX_CORR_UNITS = 6
USE_DD_RULE   = True

WARMUP_BARS   = 60          # ≥ ATR_PERIOD + ENTRY_BARS, si no se opera a ciegas
HEARTBEAT     = 0.5          # cadencia del bucle de eventos
MAE_STAR_N    = 1.55         # medido en el backtest, solo para el informe

print(f"  Señal   : fade de ruptura {ENTRY_BARS} barras, confirmación={REQUIRE_CONF}")
print(f"  Riesgo  : stop {STOP_N:.0f}N = 1R | objetivo {TP_MODE} | vertical {MAX_HOLD_BARS} barras")
print(f"  Escalera: hasta {MAX_UNITS} unidades cada {ADD_N}N")
if MAE_STAR_N > STOP_N:
    print(f"\n  ⚠ MAE* = {MAE_STAR_N:.2f}N > stop {STOP_N:.2f}N ({MAE_STAR_N/STOP_N:.2f}R):")
    print("    el cierre temprano por MAE NO puede dispararse antes que el stop.")
    print("    Se mide y se registra, pero no cierra. Subir STOP_N lo haría caber.")
    


  Señal   : fade de ruptura 20 barras, confirmación=False
  Riesgo  : stop 1N = 1R | objetivo mid_channel | vertical 10 barras
  Escalera: hasta 4 unidades cada 0.5N

  ⚠ MAE* = 1.55N > stop 1.00N (1.55R):
    el cierre temprano por MAE NO puede dispararse antes que el stop.
    Se mide y se registra, pero no cierra. Subir STOP_N lo haría caber.


## 2. Cableado del motor

El flujo es el del README y no se altera aquí:

```
trading (cola)  →  data       emite BarEvent (barra cerrada) y TickEvent (bid/ask)
                →  strategy   consume BAR, emite SignalEvent
                →  portfolio  consume SIGNAL, dimensiona, emite OrderEvent
                →  execution  consume ORDER, manda bracket a IBKR, emite FillEvent
                →  portfolio  consume FILL, reancla al precio real, emite PortfolioEvent
                →  trading    enruta el PortfolioEvent al monitor
```

Dos hilos, como en `trading/trading.py`: uno para el bucle de eventos y otro para el streaming.
El notebook **no contiene lógica de estrategia**; solo construye los componentes y observa.

In [3]:
# ── data ────────────────────────────────────────────────────────────────────
events = queue.Queue()

prices = IBKRPriceHandler(
    SYMBOLS, events, ib=ib, exchange=EXCHANGE, currency=CURRENCY,
    bar_size=BAR_SIZE, use_rth=USE_RTH, what_to_show=WHAT_TO_SHOW,
    warmup_bars=WARMUP_BARS, account=ACCOUNT,
)

ib.RequestTimeout = 45   # antes de prices.warmup(); vuelve a 20 después si quieres
prices.warmup()


prices.warmup()
SPECS = prices.specs
for s in SYMBOLS:
    h = prices.history[s]
    print(f"  {s}: {prices.specs[s]['localSymbol']}  mult={prices.specs[s]['multiplier']} "
          f"tick={prices.specs[s]['minTick']}  {len(h)} barras  "
          f"({h.index[0]:%m-%d %H:%M} → {h.index[-1]:%m-%d %H:%M})")

ib.RequestTimeout = 20

# ── strategy ────────────────────────────────────────────────────────────────
strategy = MeanReversionFadeStrategy(
    SYMBOLS, 
    events, 
    price_handler=prices, 
    entry_bars=ENTRY_BARS,
    atr_period=ATR_PERIOD, 
    signal_mode=SIGNAL_MODE,
    require_confirmation=REQUIRE_CONF, 
    confirm_mode=CONFIRM_MODE,
    allow_short=ALLOW_SHORT,
)

# ── execution ───────────────────────────────────────────────────────────────
execution = create_execution_handler(
    broker="ibkr",
    events_queue=events,
    ib=ib,
    contracts=prices.contracts, 
    specs=SPECS, account=ACCOUNT,
)

# ── portfolio ───────────────────────────────────────────────────────────────
portfolio = IBKRFuturesPortfolio(
    prices, 
    events, 
    home_currency=CURRENCY, 
    equity=EQUITY,
    risk_per_trade=Decimal(str(RISK_PER_UNIT)), 
    backtest=False, 
    specs=SPECS,
    stop_n=STOP_N, 
    add_n=ADD_N, 
    max_units=MAX_UNITS,
    max_hold_bars=MAX_HOLD_BARS, 
    tp_mode=TP_MODE,
    max_gross_lev=MAX_GROSS_LEV,
    max_dir_units=MAX_DIR_UNITS,
    max_corr_units=MAX_CORR_UNITS, 
    use_dd_rule=USE_DD_RULE,
    execution=execution, strategy=strategy,
)

n_now = strategy._n(strategy.history[SYMBOLS[0]])
qty_now = portfolio.calc_risk_position_size(SYMBOLS[0], n=n_now)
print(f"\n  N actual: {n_now:,.1f} puntos → 1R = {STOP_N*n_now:,.1f}")
print(f"  Unidad  : {qty_now} contratos → riesgo "
      f"${qty_now*STOP_N*n_now*SPECS[SYMBOLS[0]]['multiplier']:,.0f} "
      f"({RISK_PER_UNIT:.0%} del equity por construcción)")

# ── reconciliación con el bróker ─────────────────────────────────────────────
# portfolio.positions nace vacío pase lo que pase en la cuenta. Si ya hay una
# posición viva (una sesión anterior, un kernel reiniciado a media jornada),
# sin esto el monitor diría "sin posiciones" con una posición real abierta, y
# la estrategia podría intentar abrir una segunda encima de ella. Se llama
# UNA vez, antes de arrancar los hilos.
portfolio.reconcile_from_broker(ib, prices.contracts, account=ACCOUNT)
if portfolio.positions:
    print(f"\n  ⚠ Reconciliadas {len(portfolio.positions)} posición(es) ya "
          f"abiertas en el bróker: {list(portfolio.positions.keys())}")
    print("    Canal/objetivo perdidos para esas posiciones: solo saldrán "
          "por stop o barrera vertical. Revisa el log si alguna no tiene "
          "stop detectado — esa queda sin barrera automática.")
else:
    print("\n  Sin posiciones abiertas en el bróker al arrancar.")


  MBT: MBTU6  mult=0.1 tick=5.0  898 barras  (09-04 13:30 → 09-08 20:57)

  N actual: 37.8 puntos → 1R = 37.8
  Unidad  : 933 contratos → riesgo $3,531 (1% del equity por construcción)

  Sin posiciones abiertas en el bróker al arrancar.


## 3. Journal de la jornada

Una línea por decisión, append-only y con flush. Si el proceso muere, lo escrito hasta ese
instante sigue siendo la verdad reconstruible. El monitor recibe los `PortfolioEvent` que emite
el portfolio en cada cambio de estado, así que no tiene que hurgar en sus estructuras internas
mientras el hilo de precios las está mutando.

In [4]:
SESSION_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
JOURNAL = JOURNAL_DIR / f"mr_intraday_{'_'.join(SYMBOLS)}_{SESSION_ID}.jsonl"
STATE = STATE_DIR / f"live_state_mr_{'_'.join(SYMBOLS)}.json"


class Monitor(object):
    """Recibe PortfolioEvent, escribe el journal y guarda la última foto.

    También es el punto de enganche de `trade()` (qsCrypto/trading/trading.py)
    para dos señales de salud que antes no existían: `heartbeat()` marca cada
    vuelta del bucle de eventos (se llama exista o no un evento que despachar,
    así que su antigüedad delata un hilo colgado incluso si sigue "vivo" para
    `threading`) y `record_error()` cuenta los eventos que el motor tuvo que
    descartar por una excepción, en vez de dejar que el hilo muriera en
    silencio como pasaba antes.
    """

    def __init__(self, journal_path, state_path):
        self.journal_path = journal_path
        self.state_path = state_path
        self.last = []
        self.history = []
        self.errors = []
        self.last_heartbeat = None
        self.lock = threading.Lock()

    def heartbeat(self):
        self.last_heartbeat = datetime.now(timezone.utc)

    def record_error(self, event_type):
        with self.lock:
            self.errors.append((datetime.now(timezone.utc), event_type))
        self.write("dispatch_error", event_type=str(event_type))

    def write(self, evento, **extra):
        row = {"ts": datetime.now(timezone.utc).isoformat(),
               "session": SESSION_ID, "evento": evento}
        row.update(extra)
        with self.lock:
            with open(self.journal_path, "a") as f:
                f.write(json.dumps(row, default=str) + "\n")
                f.flush()
        return row

    def record(self, event):
        """Lo llama trade() al recibir un PORTFOLIO."""
        with self.lock:
            self.last = event.snapshot
        self.history.append((datetime.now(timezone.utc), event.reason,
                             len(event.snapshot)))
        self.write("portfolio", reason=event.reason, balance=event.balance,
                   positions=event.snapshot)
        # Escritura atómica: un lector nunca ve un JSON a medias.
        tmp = self.state_path.with_suffix(".tmp")
        with open(tmp, "w") as f:
            json.dump({"ts": datetime.now(timezone.utc).isoformat(),
                       "session": SESSION_ID, "reason": event.reason,
                       "balance": event.balance, "positions": event.snapshot},
                      f, indent=2, default=str)
        os.replace(tmp, self.state_path)


monitor = Monitor(JOURNAL, STATE)
monitor.write("session_start", symbols=SYMBOLS, equity=float(EQUITY),
              account=ACCOUNT, stop_n=STOP_N, entry_bars=ENTRY_BARS,
              risk=RISK_PER_UNIT, mae_star_n=MAE_STAR_N,
              mae_fits_in_stop=bool(MAE_STAR_N <= STOP_N),
              specs={s: {k: v for k, v in SPECS[s].items() if k != "expiry"}
                     for s in SYMBOLS})
print(f"  Journal : {JOURNAL}")
print(f"  Estado  : {STATE}")


  Journal : /Users/diegoochoa/Projects/Intraday_MR_BTC/journals/mr_intraday_MBT_20260908T210745Z.jsonl
  Estado  : /Users/diegoochoa/Projects/Intraday_MR_BTC/output/live_state_mr_MBT.json


## 4. Arrancar la sesión

⚠️ **Esta celda abre posiciones reales en la cuenta paper.** `RUN_LIVE = False` deja todo montado
sin enviar nada, que es como conviene arrancar la primera vez.

In [5]:
RUN_LIVE = True       # ponlo a True para operar

if not RUN_LIVE:
    print("  RUN_LIVE=False — el motor no arranca. Nada se envía a IBKR.")
else:
    trade_thread = threading.Thread(
        target=trade,
        args=(events, strategy, portfolio, execution, HEARTBEAT, monitor),
        daemon=True, name="trade-loop",
    )
    price_thread = threading.Thread(
        target=prices.stream_to_queue, daemon=True, name="ibkr-stream",
    )
    trade_thread.start()
    price_thread.start()
    print("  Motor arrancado. Hilos:", [t.name for t in (trade_thread, price_thread)])
    print("  Ejecuta la celda del monitor cuantas veces quieras.")


  Motor arrancado. Hilos: ['trade-loop', 'ibkr-stream']
  Ejecuta la celda del monitor cuantas veces quieras.


## 5. Monitor en vivo — se repinta solo cada 30s

Bloquea y se repinta cada `MONITOR_INTERVAL_S` segundos (`clear_output` + redibujo). Tres capas
de evidencia, cada una capaz de delatar un fallo que las otras no ven:

- **Feed de precios** — alimentado de una suscripción **propia** de esta celda
  (`ib.reqMktData`), no del caché `prices.prices` que mantiene el hilo `ibkr-stream`. Si ese
  caché se queda quieto porque el hilo de fondo se atascó, esta tabla lo sigue mostrando porque
  vuelve a preguntarle a IBKR en cada vuelta. Columnas: `last_tick_px`/`tick_age_s` (streaming
  propio), `last_bar_close`/`bar_age_s` (última barra de 30 min cerrada) y `stream_pump_age_s`
  (hace cuánto bombeó el hilo de fondo el loop de `ib_async` — si crece sin parar mientras
  `tick_age_s` se mantiene bajo, el hilo está colgado aunque IBKR siga entregando datos).
- **Pulso del motor** — `trade-loop` marca un pulso en cada vuelta de su bucle, exista o no un
  evento que despachar; su antigüedad delata un hilo colgado incluso cuando `threading` todavía
  lo reporta "vivo". Los errores de despacho capturados (antes mataban el hilo en silencio) se
  cuentan aparte.
- **Cruce con el bróker** — cada vuelta compara `portfolio.snapshot()` contra `ib.positions()`
  en vivo. Si difieren, imprime un aviso con el comando exacto para resincronizar
  (`portfolio.reconcile_from_broker(...)`, ya invocado una vez al construir el portfolio en la
  celda 2 para capturar cualquier posición que ya estuviera abierta al arrancar).

Debajo, las posiciones abiertas con las mismas columnas de antes:

- **`mae_r` / `mfe_r`** — excursión adversa y favorable en unidades de R, donde `R = |entrada −
  stop inicial|`.
- **`mae_budget`** — fracción del presupuesto de MAE consumida. Con `STOP_N = 1.0` el umbral
  queda fuera del stop, así que esta columna nunca llega a 1,00: el stop cierra antes.
- **`dist_stop_r`** — cuánto queda hasta el stop, en R.
- **`pnl_r`** — resultado no realizado en R.
- **`stop`** — nivel de stop vigente (se mueve con la escalera de adds).

**Balance estimado** — `portfolio.balance` (realizado) + no realizado agregado de las posiciones
abiertas. No es el PnL oficial de la cuenta IBKR: el motor no suscribe `reqPnL`, así que esto es
la contabilidad propia del portfolio, no la de TWS.

Debajo, los trades ya cerrados con su excursión final, y los contadores de rechazo.

**Para detener el monitor**: interrumpe el kernel (botón de stop / `Kernel → Interrupt`). El
motor (`trade-loop`, `ibkr-stream`) sigue corriendo en segundo plano — solo se detiene el
redibujo. Vuelve a ejecutar esta celda para retomarlo.

In [6]:
from IPython.display import display, clear_output

# 30s daba una sensación de "congelado" entre redibujos — se baja a un valor
# casi en vivo. No es literalmente "cada tick" (con MBT pueden llegar varios
# ticks por segundo y redibujar así saturaría la salida y el propio ib_async);
# 2s es lo bastante rápido para ver el precio moverse sin ahogar el notebook.
MONITOR_INTERVAL_S = 2

# Suscripción PROPIA del monitor, independiente del caché prices.prices. Ese
# caché lo mantiene vivo el hilo `ibkr-stream`; si ese hilo se congela (ver
# `IBKRPriceHandler.stream_to_queue`), el caché se queda quieto sin que nada
# lo delate. Aquí se le pregunta a IBKR directamente en cada vuelta del
# monitor, así que el heartbeat no depende de que el otro hilo siga sano.
# `globals()` evita re-suscribirse (y agotar líneas de datos) si la celda se
# re-ejecuta varias veces en la misma sesión de kernel.
if "_MONITOR_TICKERS" not in globals():
    _MONITOR_TICKERS = {s: ib.reqMktData(prices.contracts[s], "", False, False)
                        for s in SYMBOLS}


def _fmt(x, spec=".2f"):
    """Formatea o devuelve '—'. Una posición reconciliada desde el bróker
    (ver `reconcile_from_broker`) puede llegar sin stop/N/canal detectados,
    y esas columnas quedan en None a propósito en vez de mentir con un cero
    — pero un `f"{None:.2f}"` revienta con TypeError y tumba TODO el bucle
    del monitor. Nunca formatear una columna del snapshot sin pasar por esto.
    """
    return "—" if x is None else format(x, spec)


def build_monitor_df():
    """DataFrame de posiciones abiertas, o None si no hay ninguna."""
    with monitor.lock:
        rows = list(monitor.last) if monitor.last else portfolio.snapshot()
    if not rows:
        return None
    df = pd.DataFrame(rows)
    df["mae_budget"] = df["mae_r"] / (MAE_STAR_N / STOP_N)
    cols = ["pair", "side", "qty", "units", "entry", "price", "stop", "n",
            "r_unit", "pnl_r", "mae_r", "mae_price", "mae_budget",
            "mfe_r", "dist_stop_r", "bars", "pnl_base"]
    return df[[c for c in cols if c in df.columns]].round(5)


def feed_heartbeat():
    """Una fila por símbolo, alimentada del STREAMING en vivo, no del caché.

    - `last_tick_px`/`tick_age_s`: último precio y su antigüedad, leídos del
      ticker propio de esta celda (`_MONITOR_TICKERS`), pedido a IBKR de
      nuevo en cada vuelta — no del diccionario `prices.prices` que mantiene
      el hilo de fondo.
    - `stream_pump_age_s`: hace cuánto que `ibkr-stream` bombeó el loop de
      ib_async por última vez (`prices.last_pump_at`). Si esto crece sin
      parar mientras `tick_age_s` se mantiene bajo, el hilo de fondo está
      colgado pero IBKR sigue entregando datos — hay que reiniciar el hilo.
    - `last_bar_close`/`bar_age_s`: la última barra de 30 min ya cerrada.
    """
    now = datetime.now(timezone.utc)
    rows = []
    for s in SYMBOLS:
        tk = _MONITOR_TICKERS[s]
        last = tk.last if tk.last is not None and tk.last == tk.last else None
        bid, ask = tk.bid, tk.ask
        mid = ((bid + ask) / 2 if bid is not None and ask is not None
               and bid == bid and ask == ask else None)
        px = last if last is not None else mid
        tick_time = tk.time  # datetime tz-aware de ib_async, o None
        tick_age_s = (now - tick_time).total_seconds() if tick_time else None

        h = prices.history.get(s)
        if h is not None and len(h):
            last_bar_ts = h.index[-1]          # naive, en UTC (ver warmup())
            last_bar_close = h.iloc[-1].get("close")
            bar_age_s = (now.replace(tzinfo=None) - last_bar_ts).total_seconds()
        else:
            last_bar_ts = last_bar_close = bar_age_s = None

        pump_at = getattr(prices, "last_pump_at", None)
        pump_age_s = (now - pump_at).total_seconds() if pump_at else None

        rows.append({"symbol": s, "last_tick_px": px, "tick_age_s": tick_age_s,
                     "last_bar_close": last_bar_close, "last_bar_ts": last_bar_ts,
                     "bar_age_s": bar_age_s, "stream_pump_age_s": pump_age_s})
    return pd.DataFrame(rows)


def broker_cross_check():
    """Compara portfolio.snapshot() contra ib.positions() EN VIVO.

    Es la comprobación que habría delatado antes el desacople real: el
    portfolio decía "sin posiciones" mientras la cuenta tenía una abierta.
    `reconcile_from_broker` (celda de construcción del portfolio) cierra el
    hueco al arrancar; esto vigila que no se vuelva a abrir mientras corre.
    """
    broker_syms = set()
    for p in ib.positions():
        if float(p.position) == 0 or (ACCOUNT and p.account != ACCOUNT):
            continue
        for s, c in prices.contracts.items():
            if c.conId == p.contract.conId:
                broker_syms.add(s)
    local_syms = set(portfolio.positions.keys())
    return broker_syms, local_syms


def show_monitor_once():
    """Un redibujo: heartbeat del feed, hilos, posiciones, cerrados, rechazos."""
    now = datetime.now(timezone.utc)
    hb_age = ((now - monitor.last_heartbeat).total_seconds()
              if monitor.last_heartbeat else None)
    pulse = f"pulso hace {hb_age:.1f}s" if hb_age is not None else "sin pulso todavía"
    print(f"  {now:%Y-%m-%d %H:%M:%S} UTC — "
          f"trade-loop {'vivo' if trade_thread.is_alive() else 'MUERTO'} ({pulse}) | "
          f"ibkr-stream {'vivo' if price_thread.is_alive() else 'MUERTO'}"
          + (f" | errores despacho: {len(monitor.errors)}" if monitor.errors else ""))

    print("\n  Feed de precios (streaming en vivo):")
    display(feed_heartbeat().round(5))

    broker_syms, local_syms = broker_cross_check()
    if broker_syms != local_syms:
        print(f"\n  ⚠⚠ DESACOPLE con el bróker: cuenta tiene {sorted(broker_syms)} "
              f"pero el portfolio local ve {sorted(local_syms)}. "
              f"Ejecuta portfolio.reconcile_from_broker(ib, prices.contracts, "
              f"account=ACCOUNT) para resincronizar.")

    df = build_monitor_df()
    print()
    if df is None:
        print("  Sin posiciones abiertas (local).")
        unrealized = 0.0
    else:
        display(df)
        for _, r in df.iterrows():
            print(f"  {r['pair']} {r['side']} x{int(r['qty'])} | "
                  f"P&L {_fmt(r['pnl_r'], '+.2f')}R | "
                  f"MAE {_fmt(r['mae_r'])}R "
                  f"({_fmt(r['mae_budget'], '.0%')} del umbral) | "
                  f"MFE {_fmt(r['mfe_r'])}R | "
                  f"stop a {_fmt(r['dist_stop_r'])}R | "
                  f"stop vigente {_fmt(r['stop'], '.1f')}")
            if r["stop"] is None or r["n"] is None:
                print(f"    ⚠ {r['pair']}: posición reconciliada sin stop y/o N "
                      f"detectado — revisa TWS, puede estar sin barrera automática.")
        unrealized = df["pnl_base"].sum() if "pnl_base" in df.columns else 0.0

    print(f"\n  Balance estimado: {float(portfolio.balance):,.2f} realizado "
          f"+ {unrealized:,.2f} no realizado = "
          f"{float(portfolio.balance) + unrealized:,.2f}")

    closed = portfolio.closed_excursions
    print(f"\n  Trades cerrados: {len(closed)}")
    if closed:
        cdf = pd.DataFrame(closed)
        cols = [c for c in ["currency_pair", "position_type", "units",
                            "entry_anchor", "exit_price", "pnl", "mae_r",
                            "mfe_r", "mae_price", "mfe_price"] if c in cdf.columns]
        display(cdf[cols].round(5))
        if "mae_r" in cdf and cdf["mae_r"].notna().any():
            win = cdf["pnl"] > 0
            print(f"  MAE media ganadores: {cdf.loc[win,'mae_r'].mean():.2f}R | "
                  f"perdedores: {cdf.loc[~win,'mae_r'].mean():.2f}R")

    print(f"\n  Rechazos estrategia: {strategy.rejected}")
    print(f"  Rechazos portfolio : {portfolio.rejected}")
    if monitor.errors:
        print(f"  Últimos errores del motor: {monitor.errors[-3:]}")
    if IB_ERRORS:
        print(f"  Errores TWS        : {IB_ERRORS[-3:]}")
    print(f"\n  Log detallado del motor (BAR/SEÑAL/rechazos): "
          f"tail -f {LOG_FILE}")


def monitor_loop(interval=MONITOR_INTERVAL_S, iterations=None):
    """Redibuja show_monitor_once() cada `interval` segundos.

    Bloquea hasta que se interrumpe el kernel o se agotan `iterations` (si se
    pasa, útil para pruebas cortas sin tener que interrumpir). El motor sigue
    vivo en sus propios hilos aunque este bucle se detenga: solo deja de
    redibujarse. `ib.sleep(interval)` en vez de `time.sleep`: así este mismo
    hilo del notebook también bombea el loop de ib_async mientras espera, en
    vez de quedarse completamente pasivo.
    """
    i = 0
    try:
        while iterations is None or i < iterations:
            clear_output(wait=True)
            show_monitor_once()
            i += 1
            ib.sleep(interval)
    except KeyboardInterrupt:
        clear_output(wait=True)
        show_monitor_once()
        print("\n  Monitor detenido manualmente. El motor sigue corriendo en segundo plano.")



monitor_loop()

  2026-09-08 21:08:04 UTC — trade-loop vivo (pulso hace 0.5s) | ibkr-stream vivo

  Feed de precios (streaming en vivo):


,symbol,last_tick_px,tick_age_s,last_bar_close,last_bar_ts,bar_age_s,stream_pump_age_s
0,MBT,78660.0,0.75905,78645.0,2026-09-08 20:57:00,664.77071,None



  Sin posiciones abiertas (local).

  Balance estimado: 353,172.89 realizado + 0.00 no realizado = 353,172.89

  Trades cerrados: 0

  Rechazos estrategia: {'sin_confirmar': 0, 'plana': 0, 'sin_n': 0, 'posicion_abierta': 0}
  Rechazos portfolio : {'qty_cero': 0, 'apalancamiento': 0, 'topes': 0}

  Log detallado del motor (BAR/SEÑAL/rechazos): tail -f /Users/diegoochoa/Projects/Intraday_MR_BTC/output/live_engine.log

  Monitor detenido manualmente. El motor sigue corriendo en segundo plano.


### Esta celda permite forzar una posición.
Obtenemos los canales vigentes con el paso 1, el paso 2 elimina la necesidad de confirmación

In [7]:
from qsCrypto.event.event import BarEvent
import pandas as pd, time

sym = SYMBOLS[0]
ent_hi, ent_lo, mid = strategy._channel(strategy.history[sym])
t0 = pd.Timestamp.utcnow().tz_localize(None)
step = pd.Timedelta(seconds=30)
print(f"Canal actual: [{ent_lo:.5f}, {ent_hi:.5f}], mid={mid:.5f}")




Canal actual: [78620.00000, 78750.00000], mid=78685.00000


In [ ]:
# 1) ruptura ALCISTA clara, por encima de ent_hi
brk = ent_hi + 20
events.put(BarEvent(sym, t0, ent_hi, brk, ent_hi, brk, 1.0, complete=True))
time.sleep(2)

# 2) confirmación: cierra DENTRO del canal (si REQUIRE_CONF=True)
if strategy.require_confirmation:
    t1 = t0 + step
    confirm_close = mid  # bien adentro del canal
    events.put(BarEvent(sym, t1, brk, brk, confirm_close, confirm_close, 1.0, complete=True))
    time.sleep(2)

print("Rechazos estrategia:", strategy.rejected)
print("Pendiente:", strategy.pending[sym])

El paso 3 permite inyectar manualmente una señal en la cola de eventos para detonar una colocación de ordenes 

In [ ]:
from qsCrypto.event.event import SignalEvent
import pandas as pd

from decimal import Decimal
portfolio.risk_per_trade = Decimal("0.00001")   # ~13x más chico que 0.001, para compensar N=32 vs N≈424 en velas de 1 minuto

# 2) inyecta una señal de ejecución direcamente en la cola. Los precios deben permitir una engtrada. 
sig = SignalEvent("MBT", "market", "sell", pd.Timestamp.utcnow(),
                   n=10, level=78620, mid=78500)
events.put(sig)
import time; time.sleep(2)

print("Posiciones:", list(portfolio.positions.keys()))
print("Rechazos portfolio:", portfolio.rejected)



puede pasarse esta linea en el terminal para ver el flujo de ticks en vivo : `tail -f output/live_engine.log`

### Esta celda permite verificar que el flujo continua conectado 

In [ ]:
print("trade-loop :", trade_thread.is_alive())
print("ibkr-stream:", price_thread.is_alive())
print("ib conectado:", ib.isConnected())
print("último pulso:", monitor.last_heartbeat)


In [ ]:
prices.ib.pendingTickersEvent -= prices._on_pending_tickers
for pair, bars in list(prices._bar_subs.items()):
    prices.ib.cancelHistoricalData(bars)
for pair in list(prices._tickers):
    prices.ib.cancelMktData(prices.contracts[pair])
prices._bar_subs.clear(); prices._tickers.clear()
prices.subscribe()
print("Resuscrito. Verifica en 30-60s que bar_age_s baja en el monitor.")


## 6. Cierre de la jornada

Cancela lo vivo, aplana y cierra la conexión. `flatten()` cancela **antes** de aplanar: al revés,
el stop adjunto seguiría vivo tras el cierre y quedaría como una orden huérfana capaz de abrir
posición contraria.

In [8]:
def kill_all(reason="fin_de_jornada"):
    for s in SYMBOLS:
        try:
            flat = execution.flatten(s, reason=reason)
            monitor.write("kill", symbol=s, flat=flat, reason=reason)
            print(f"  {s}: aplanado {flat:+d}")
        except Exception as exc:
            print(f"  {s}: fallo al aplanar — {exc}")


kill_all()          # descomenta para aplanar todo ahora mismo

monitor.write("session_end", closed=len(portfolio.closed_excursions),
               balance=float(portfolio.balance))
prices.stop(); ib.disconnect(); print("  Sesión cerrada.")


  MBT: aplanado +0
  Sesión cerrada.
